In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

# Adjust base path to typical Kaggle environment
base_path = "./input/champs-scalar-coupling"

train_path = os.path.join(base_path, "train.csv")
test_path = os.path.join(base_path, "test.csv")
structures_path = os.path.join(base_path, "structures.csv")

train_df = pd.read_csv(
    train_path,
    usecols=[
        "id",
        "molecule_name",
        "atom_index_0",
        "atom_index_1",
        "type",
        "scalar_coupling_constant",
    ],
    dtype={
        "id": np.int32,
        "molecule_name": "category",
        "atom_index_0": np.int16,
        "atom_index_1": np.int16,
        "type": "category",
        "scalar_coupling_constant": np.float32,
    },
)

test_df = pd.read_csv(
    test_path,
    usecols=["id", "molecule_name", "atom_index_0", "atom_index_1", "type"],
    dtype={
        "id": np.int32,
        "molecule_name": "category",
        "atom_index_0": np.int16,
        "atom_index_1": np.int16,
        "type": "category",
    },
)

structures = pd.read_csv(
    structures_path,
    usecols=["molecule_name", "atom_index", "atom", "x", "y", "z"],
    dtype={
        "molecule_name": "category",
        "atom_index": np.int16,
        "atom": "category",
        "x": np.float32,
        "y": np.float32,
        "z": np.float32,
    },
)

structures0 = structures.rename(
    columns={"atom": "atom_0", "x": "x0", "y": "y0", "z": "z0"}
).set_index(["molecule_name", "atom_index"])
structures1 = structures.rename(
    columns={"atom": "atom_1", "x": "x1", "y": "y1", "z": "z1"}
).set_index(["molecule_name", "atom_index"])


def add_atom_info(df):
    df = df.merge(
        structures0,
        left_on=["molecule_name", "atom_index_0"],
        right_index=True,
        how="left",
    )
    df = df.merge(
        structures1,
        left_on=["molecule_name", "atom_index_1"],
        right_index=True,
        how="left",
    )
    return df


train_full = add_atom_info(train_df)
test_full = add_atom_info(test_df)

atom_number = {
    "H": 1,
    "C": 6,
    "N": 7,
    "O": 8,
    "F": 9,
    "Cl": 17,
    "Br": 35,
    "I": 53,
    "S": 16,
    "P": 15,
}


def get_num(sym):
    return atom_number.get(sym, 0)


for col in ["atom_0", "atom_1"]:
    train_full[f"{col}_num"] = train_full[col].map(get_num).astype(np.int8)
    test_full[f"{col}_num"] = test_full[col].map(get_num).astype(np.int8)

train_full["atom_sum_num"] = train_full["atom_0_num"] + train_full["atom_1_num"]
test_full["atom_sum_num"] = test_full["atom_0_num"] + test_full["atom_1_num"]

train_full["distance"] = np.sqrt(
    (train_full["x0"] - train_full["x1"]) ** 2
    + (train_full["y0"] - train_full["y1"]) ** 2
    + (train_full["z0"] - train_full["z1"]) ** 2
).astype(np.float32)
test_full["distance"] = np.sqrt(
    (test_full["x0"] - test_full["x1"]) ** 2
    + (test_full["y0"] - test_full["y1"]) ** 2
    + (test_full["z0"] - test_full["z1"]) ** 2
).astype(np.float32)

train_full["type_code"] = train_full["type"].cat.codes
type_uniques = train_full["type"].cat.categories
test_full["type_code"] = test_full["type"].cat.codes

feature_cols = ["distance", "atom_0_num", "atom_1_num", "atom_sum_num", "type_code"]
imputer = SimpleImputer(strategy="median")
imputer.fit(train_full[feature_cols])
train_full[feature_cols] = imputer.transform(train_full[feature_cols])
test_full[feature_cols] = imputer.transform(test_full[feature_cols])


def make_combo_key(df):
    a0 = df["atom_0"].astype(str).values
    a1 = df["atom_1"].astype(str).values
    min_atom = np.where(a0 < a1, a0, a1)
    max_atom = np.where(a0 < a1, a1, a0)
    return df["type"].astype(str).values + "_" + min_atom + "_" + max_atom


# Convert to pandas Series before casting to categorical to avoid TypeError
train_full["combo_key"] = pd.Series(
    make_combo_key(train_full), index=train_full.index, dtype="category"
)
test_full["combo_key"] = pd.Series(
    make_combo_key(test_full), index=test_full.index, dtype="category"
)

type_means_full = train_full.groupby("type")["scalar_coupling_constant"].mean()
global_mean_full = train_full["scalar_coupling_constant"].mean()
combo_means_full = train_full.groupby("combo_key")["scalar_coupling_constant"].mean()

train_split, val_split = train_test_split(
    train_full,
    test_size=0.2,
    random_state=42,
    stratify=train_full["type"],
)

X_train = train_split[feature_cols].values.astype(np.float32)
y_train = train_split["scalar_coupling_constant"].values.astype(np.float32)

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=42,
    min_samples_leaf=1,
    max_features="sqrt",
)
rf.fit(X_train, y_train)

combo_fallback = (
    val_split["combo_key"]
    .map(combo_means_full)
    .fillna(val_split["type"].map(type_means_full))
    .fillna(global_mean_full)
)

X_val = val_split[feature_cols].values.astype(np.float32)
model_pred = rf.predict(X_val)


def log_mae_series(true, pred, types):
    mae_per_type = true.groupby(types).apply(
        lambda g: mean_absolute_error(g["scalar_coupling_constant"], pred.loc[g.index])
    )
    return np.log(mae_per_type).mean()


fallback_series = pd.Series(combo_fallback.values, index=val_split.index)
fallback_log_mae = log_mae_series(val_split, fallback_series, "type")

model_series = pd.Series(model_pred, index=val_split.index)
model_log_mae = log_mae_series(val_split, model_series, "type")

blend_weight = 0.0
if model_log_mae < fallback_log_mae:
    blend_weight = 0.2

val_pred_series = pd.Series(
    blend_weight * model_pred + (1 - blend_weight) * combo_fallback.values,
    index=val_split.index,
)

final_log_mae = log_mae_series(val_split, val_pred_series, "type")
print(f"Validation log‑MAE (lower is better): {final_log_mae:.5f}")

# ---- Test predictions ----
test_combo_fallback = (
    test_full["combo_key"]
    .map(combo_means_full)
    .fillna(test_full["type"].map(type_means_full))
    .fillna(global_mean_full)
)

X_test = test_full[feature_cols].values.astype(np.float32)
test_model_pred = rf.predict(X_test)

test_pred = (
    blend_weight * test_model_pred + (1 - blend_weight) * test_combo_fallback.values
)

submission = pd.DataFrame(
    {"id": test_full["id"], "scalar_coupling_constant": test_pred}
)



FileNotFoundError: [Errno 2] No such file or directory: './input/champs-scalar-coupling/train.csv'

In [2]:
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False, float_format="%.6f")
print(f"Submission written to {submission_path}")

NameError: name 'submission' is not defined